# Automated Drug Prescription System with Biomedical BERT Embeddings and Similarity Search

This notebook builds a **retrieval-based drug recommendation system**. Drug and disease descriptions are embedded with domain-specific BERT models (**PubMedBERT** and **ClinicalBERT**). A disease or symptom query is then matched to the most similar entries with three similarity-search techniques, and their speed and returned results are compared.

| | |
|---|---|
| **Dataset 1** | [Medicine Recommendation System Dataset](https://www.kaggle.com/datasets/noorsaeed/medicine-recommendation-system-dataset): disease → symptoms and disease → medications tables (`symtoms_df.csv`, `medications.csv`) |
| **Dataset 2** | [34,000 Medicine Details](https://www.kaggle.com/datasets/samoyedzzzzz/34000-medicine-details): about 34k drugs scraped from 1mg.com with their names and uses |
| **Embedding models** | `neuml/pubmedbert-base-embeddings` (mean pooling) and `medicalai/ClinicalBERT` ([CLS] token) |
| **Search techniques** | Locality-Sensitive Hashing (LSH, `pyLSHash`), SimHash on raw text, and exact L2 nearest-neighbour search with FAISS (`IndexFlatL2`) |
| **Output** | Top-k recommended medications for a held-out disease or symptom query |

**Notebook outline:** Installing & Importing Required Packages → Loading Model → Testing Tokenizer & Output Embeddings → Generating Embeddings (data download, embedding function, data preparation) → Recommendation (LSH, SimHash and FAISS on both datasets).

> **Note on running this notebook.** This is the original project notebook, kept as it was; only these explanatory cells have been added. It was run on Google Colab with data and embeddings stored on Google Drive (`/content/drive/MyDrive/BDA_Project/`). Its cells were executed across several sessions with different models and datasets loaded, so some outputs come from earlier runs. The notes in each section point out where this matters.

# Installing & Importing Required Packages

Basharat Ali        452373
Syed Abdul Hanan    450189

## Mount Google Drive

Mounts Google Drive in Colab so the datasets and the pre-computed embedding files (`.npy`) persist between sessions.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Dependencies and imports

Installs the libraries that Colab doesn't provide by default:

- **kagglehub**: downloads the datasets from Kaggle.
- **pyLSHash**: provides `LSHash` for locality-sensitive hashing of embeddings and `SimHash`/`hamming` for text fingerprinting.
- **faiss-cpu**: Facebook AI Similarity Search, used here for exact L2 nearest-neighbour search.

The cell then imports Hugging Face `transformers` to load the BERT models, along with pandas, NumPy and PyTorch.

In [ ]:
!pip install kagglehub
!pip install pyLSHash
!pip install faiss-cpu


import kagglehub
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModel
import shutil
import os
import pandas as pd
import numpy as np
from pyLSHash import LSHash
import faiss
import re
import torch
from time import time
from pyLSHash import SimHash, hamming


# Loading Model

## PUBMED BERT

Loads **PubMedBERT** (`neuml/pubmedbert-base-embeddings`), a BERT model pre-trained on PubMed abstracts and fine-tuned to produce sentence embeddings. Its embeddings are later obtained by mean-pooling the token vectors.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("neuml/pubmedbert-base-embeddings")
model = AutoModel.from_pretrained("neuml/pubmedbert-base-embeddings")


config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

## Clinical BERT

Loads **ClinicalBERT** (`medicalai/ClinicalBERT`), a BERT model adapted to clinical text. It is loaded with a masked-language-modelling head, and the embedding is taken from the last hidden layer.

> Both models are assigned to the same `tokenizer` and `model` variables, so whichever cell ran last decides the model used by the cells below.

In [ ]:

tokenizer = AutoTokenizer.from_pretrained("medicalai/ClinicalBERT")
model = AutoModelForMaskedLM.from_pretrained("medicalai/ClinicalBERT")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


# Testing Tokenizer & Output Embeddings

## Case 1

A quick sanity check of the tokenizer and model: one sentence is tokenised, passed through the model with `output_hidden_states=True`, and the last-layer token embeddings and decoded tokens are inspected.

In [ ]:
inputs = tokenizer("Fungal infection is a common skin condition caused by fungi.", return_tensors="pt")
outputs = model(**inputs, output_hidden_states=True)


In [ ]:
hidden_states = outputs.hidden_states
last_layer_embeddings = hidden_states[-1]

In [ ]:
inputs

{'input_ids': tensor([[  101, 16068,  6997,  8974,  1110,   170,  1887,  2241,  3879,  2416,
          1118, 17005,   119,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [ ]:
last_layer_embeddings

tensor([[[ 0.2802, -0.0495, -0.3308,  ..., -0.1048,  0.2501, -0.2742],
         [-0.2599,  0.0293,  0.0121,  ..., -0.1601,  0.0475, -0.3550],
         [-0.0697,  0.0880,  0.1570,  ..., -0.0232, -0.1372, -0.5159],
         ...,
         [ 0.1192,  0.1219, -0.2644,  ...,  0.0856,  0.0011, -0.3336],
         [-0.1192, -0.1487, -0.2510,  ..., -0.0660,  0.1240, -0.2252],
         [ 0.8265, -0.8072, -0.8359,  ..., -0.7220,  0.6970, -0.5170]]],
       grad_fn=<NativeLayerNormBackward0>)

In [ ]:
tokenizer.decode(inputs["input_ids"][0])

'[CLS] fungal infection is a common skin condition caused by fungi. [SEP]'

## Case 2

The same check on a nearly identical sentence ("condition" replaced by "disease"). The second cell extracts the **[CLS] token** embedding, a 768-dimensional vector that summarises the sentence. This is the representation used for ClinicalBERT below.

> The two cases were evidently run with different models loaded. Case 1's output includes `token_type_ids` and a ~30k-word vocabulary, consistent with PubMedBERT. Case 2's output has no `token_type_ids` and much larger token ids, consistent with ClinicalBERT.

In [ ]:
inputs = tokenizer("Fungal infection is a common skin disease caused by fungi.", return_tensors="pt")
outputs = model(**inputs, output_hidden_states=True)
hidden_states = outputs.hidden_states
last_layer_embeddings = hidden_states[-1]
inputs

{'input_ids': tensor([[   101,  41807,  17026,  55223,  10124,    169,  14624,  40564,  21911,
          19513,  10155, 107950,    119,    102]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [ ]:
cls_embedding = last_layer_embeddings[:, 0, :]
cls_embedding.shape, last_layer_embeddings.shape

(torch.Size([1, 768]), torch.Size([1, 14, 768]))

# Generating Embeddings

### Downloading Dataset From Kaggle

Downloads **Dataset 2** (34,000 Medicine Details) with `kagglehub` and moves the CSV to Google Drive.

> The download step for **Dataset 1** (`noorsaeed/medicine-recommendation-system-dataset`) isn't in this version of the notebook. Its CSVs are read from `BDA_Project/Dataset1/` on Google Drive further below.

In [ ]:
path = kagglehub.dataset_download("samoyedzzzzz/34000-medicine-details")

100%|██████████| 6.16M/6.16M [00:00<00:00, 47.0MB/s]

Extracting files...


In [ ]:
source_path = path

destination_path = "/content/drive/MyDrive/BDA_Project/Dataset2"
os.makedirs(destination_path, exist_ok=True)

for file_name in os.listdir(source_path):
    full_file_name = os.path.join(source_path, file_name)
    if os.path.isfile(full_file_name):
        shutil.move(full_file_name, destination_path)

# Embedding Generation

## Generate Function

Two helper functions turn text into fixed-length vectors:

- **`meanpooling`** averages the last-layer token embeddings, weighted by the attention mask so that padding tokens are ignored. This is the recommended pooling for the PubMedBERT sentence-embedding model.
- **`generate_embeddings`** tokenises texts in batches of 32 (truncated to 128 tokens), runs the model on GPU when one is available, and returns one 768-dimensional vector per text. `Bert='pubmed'` selects mean pooling; any other value uses the [CLS] token of the last hidden layer (ClinicalBERT). With `query=True`, each text gets the prefix *"Medication for "* so that queries are phrased like a request for a drug.

In [ ]:
def meanpooling(output, mask):
    """
    Mean Pooling - Take attention mask into account for correct averaging.
    """
    embeddings = output[0]
    mask = mask.to(embeddings.device)
    mask = mask.unsqueeze(-1).expand(embeddings.size()).float()
    return torch.sum(embeddings * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)


In [ ]:
def generate_embeddings(
    texts, tokenizer, model, query=False,
    max_length=128, batch_size=32, Bert=None):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    all_embeddings = []
    if query:
        prefix = "Medication for "
        texts = [f"{prefix}{text}" for text in texts]

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        )

        model_inputs = {
            "input_ids": inputs["input_ids"].to(device),
            "attention_mask": inputs["attention_mask"].to(device),
            "output_hidden_states": True,
            "return_dict": True,
        }

        if "token_type_ids" in inputs:
            model_inputs["token_type_ids"] = inputs["token_type_ids"].to(device)

        with torch.no_grad():
            outputs = model(**model_inputs)
        if Bert == 'pubmed':
            embeddings = meanpooling(outputs, inputs["attention_mask"]).cpu().numpy()
        else:
            embeddings = outputs.hidden_states[-1][:, 0, :].cpu().numpy()
        all_embeddings.append(embeddings)

    return np.vstack(all_embeddings)


## Data Preparation

### Dataset 2 loading

Prepares **Dataset 2** (drug name → uses):

1. Keeps only the `Name` and `Uses` columns.
2. In the raw data, several uses are concatenated without spaces (e.g. "Treatment of DepressionTreatment of Anxiety"). A regular expression splits them back into words.
3. Builds one descriptive sentence per drug: *"Drug Name is <Name> Which is used for <Uses>"*. These sentences are embedded and indexed.

In [4]:
df = pd.read_csv("/content/drive/MyDrive/BDA_Project/Dataset2/dataset.csv")
important_columns = ['Name', 'Uses']
df = df[important_columns].fillna("")

def split_side_effects(text):
    return re.findall(r'[A-Z][a-z]*|[a-z]+|\(.*?\)|\S+', text)

df['Uses'] = df['Uses'].apply(lambda text: ' '.join(split_side_effects(text)))
df['input_text'] = "Drug Name is " + df['Name'] + " Which is used for " + df['Uses']
df['input_text'].head()


,input_text
0,Drug Name is Aceloflam Plus Tablet Which is us...
1,Drug Name is Aceloflam Plus Tablet Which is us...
2,Drug Name is Aceloflam Plus Tablet Which is us...
3,Drug Name is Aceloflam Plus Tablet Which is us...
4,Drug Name is Adriamycin 50mg Injection Which i...


Keeps only drugs whose `Uses` text is shared by at least one other drug (34,109 rows remain). That way a held-out drug's use-case is very likely still represented by other drugs in the searchable data.

In [5]:
df = df.groupby('Uses').filter(lambda group: len(group) > 1)
df.shape

(34109, 3)

Shuffles the data and holds out **200 random drugs as test queries**. They're removed from the searchable data, leaving 33,909 drugs. The query text is the drug's `Uses` field, and the held-out drug serves as the "actual" drug for comparison.

In [6]:
df = df.sample(frac=1, random_state=42)
new_df = df.sample(n=200, random_state=42)
df = df.drop(new_df.index)
df.shape, new_df.shape

((33909, 3), (200, 3))

Builds the three lists used throughout: `Data` (sentences to embed and search), `Names` (the drug name returned for each entry) and `Queries` (the use-case text of the held-out drugs). The next cells inspect example entries.

In [7]:
#df2 = pd.read_csv('/content/drive/MyDrive/BDA_Project/Dataset1/Symptom-severity.csv')
Data = df['input_text'].tolist()
Names = df['Name'].tolist()
Queries = new_df['Uses'].tolist()

In [8]:
Data[50]

'Drug Name is Dard MR Tablet Which is used for Treatment of Pain due to muscle spasm'

In [9]:
Queries[50]

'Treatment of Depression Treatment of Anxiety disorder Treatment of Panic disorder'

In [10]:
Names[50]

'Dard MR Tablet'

### Dataset 1 loading

Prepares **Dataset 1** (disease → symptoms → medications):

1. Merges the symptom table and the medication table on `Disease`. The medication list, stored as a string representation of a Python list, is flattened into comma-separated text.
2. Builds one sentence per row: *"<Disease> have symptoms <S1>, <S2> and <S3> for which <first medication> is used to cure."*
3. Holds out 200 random rows as test queries, leaving 4,240 rows to search. For the queries, only the first three symptoms are kept (`Symptom_Text`), so the system has to infer the treatment from symptoms alone.
4. Uses the first listed medication as the item to recommend (`Name`). In this dataset these are drug classes such as "Antibiotics" or "Pain relievers" rather than branded products.

> This section reassigns `df`, `new_df`, `Data`, `Names` and `Queries`, overwriting the Dataset 2 variables. The two datasets are meant to be run one at a time.

In [ ]:
disease_symptoms_df = pd.read_csv('/content/drive/MyDrive/BDA_Project/Dataset1/symtoms_df.csv')
disease_medications_df = pd.read_csv('/content/drive/MyDrive/BDA_Project/Dataset1/medications.csv')
disease_medications_df['Medication'] = disease_medications_df['Medication'].apply(lambda x: ', '.join(eval(x)) if isinstance(x, str) else x)
df = pd.merge(disease_symptoms_df, disease_medications_df, on='Disease', how='inner')
df['BERT_Input'] = df.apply(
    lambda row: f"{row['Disease']} have symptoms {row['Symptom_1']}, {row['Symptom_2']} and {row['Symptom_3']} for which {row['Medication'].split(',')[0] if isinstance(row['Medication'], str) else row['Medication']} is used to cure.",
    axis=1
)


In [ ]:
df = df.sample(frac=1, random_state=42)
new_df = df.sample(n=200, random_state=42)
df = df.drop(new_df.index)
df.shape, new_df.shape

((4240, 8), (200, 8))

In [ ]:
new_df['Symptom_Text'] = new_df[['Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4']].apply(
    lambda row: ', '.join([symptom for symptom in row if pd.notnull(symptom)][:3]),
    axis=1
)

new_df = new_df[['Disease', 'Symptom_Text']]

In [ ]:
df.head()

,Unnamed: 0,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Medication,BERT_Input
2451,2718,Cervical spondylosis,back_pain,weakness_in_limbs,neck_pain,dizziness,"Pain relievers, Muscle relaxants, Physical the...","Cervical spondylosis have symptoms back_pain,..."
4084,4527,Dengue,skin_rash,chills,joint_pain,vomiting,"Antibiotics, Antipyretics, Analgesics, IV flui...","Dengue have symptoms skin_rash, chills and ..."
2589,2869,Impetigo,skin_rash,high_fever,blister,red_sore_around_nose,"Topical antibiotics, Oral antibiotics, Antisep...","Impetigo have symptoms skin_rash, high_fever..."
478,548,Paralysis (brain hemorrhage),vomiting,weakness_of_one_body_side,altered_sensorium,NaN,"Blood thinners, Clot-dissolving medications, A...",Paralysis (brain hemorrhage) have symptoms vo...
1752,1942,Varicose veins,cramps,bruising,obesity,swollen_legs,"Levothyroxine, Antithyroid medications, Beta-b...","Varicose veins have symptoms cramps, bruisin..."


In [ ]:
df['Name'] = df['Medication'].apply(lambda x: x.split(',')[0] if isinstance(x, str) else x)

In [ ]:
df['Name'].head()

,Name
2451,Pain relievers
4084,Antibiotics
2589,Topical antibiotics
478,Blood thinners
1752,Levothyroxine


In [ ]:
Data = df['BERT_Input'].tolist()
Names = df['Name'].tolist()
Queries = new_df['Symptom_Text'].tolist()

In [ ]:
Queries[0]

' high_fever,  blister,  red_sore_around_nose'

## Embeddings

Embeds the searchable sentences and the 200 queries (with the *"Medication for"* prefix) and saves both arrays to Google Drive, so the costly embedding step only has to run once. As recorded, this cell embedded **Dataset 1** (4,240 × 768 and 200 × 768) with [CLS] pooling (`Bert=None`). That means ClinicalBERT was used if the model cells ran in order.

> `os.makedirs` creates `BDA_Project/Embeddings`, but the arrays are saved to `BDA_Project/Dataset1/Embeddings`, so that folder must already exist. The code that produced the Dataset 2 PubMedBERT embeddings loaded in the next section isn't in this version of the notebook.

In [ ]:
embeddings = generate_embeddings(Data, tokenizer, model, Bert=None)
print(embeddings.shape)
query_embeddings = generate_embeddings(Queries, tokenizer, model, query=True, Bert=None)
print(query_embeddings.shape)

(4240, 768)
(200, 768)


In [ ]:
os.makedirs('/content/drive/MyDrive/BDA_Project/Embeddings', exist_ok=True)
np.save("/content/drive/MyDrive/BDA_Project/Dataset1/Embeddings/embeddings.npy", embeddings)
np.save("/content/drive/MyDrive/BDA_Project/Dataset1/Embeddings/query_embeddings.npy", query_embeddings)



# Recommendation

## Dataset 2 Emb REC

Loads the pre-computed **PubMedBERT** embeddings for Dataset 2: 33,909 drug sentences and 200 queries, each 768-dimensional.

In [11]:
embeddings = np.load("/content/drive/MyDrive/BDA_Project/Embeddings/embeddings_pubmedbert.npy")
query_embeddings = np.load("/content/drive/MyDrive/BDA_Project/Embeddings/query_embeddings_pubmedbert.npy")

print(f"Loaded embeddings shape: {embeddings.shape}")
print(f"Loaded query embeddings shape: {query_embeddings.shape}")

Loaded embeddings shape: (33909, 768)
Loaded query embeddings shape: (200, 768)


### LSH

`LSHIndex` wraps `pyLSHash.LSHash`:

- **Indexing:** each embedding is hashed into an 8-bit bucket (`hash_size=8`) using random projections, and the drug name is stored as extra data. Similar vectors tend to land in the same bucket.
- **Querying:** only the candidates in the query's bucket are ranked by distance, which avoids comparing the query to the whole dataset. Each result's distance is converted to a similarity score with `1 / (1 + distance)`.

In [12]:
class LSHIndex:
    def __init__(self, hash_size=8, input_dim=None):
        """
        Initialize the LSHIndex.

        Parameters
        ----------
        hash_size : int, optional
            The hash size for LSHash, by default 8.
        input_dim : int, optional
            The dimension of the input embeddings.
            If None, it will be inferred from the given embeddings.
        """
        self.hash_size = hash_size
        self.input_dim = input_dim
        self.lsh = None
        self.texts = None

    def build_index(self, embeddings, texts):
        """
        Build the LSH index for the provided embeddings.

        Parameters
        ----------
        embeddings : np.ndarray
            A 2D array of shape (N, D), where N is the number of items and D is the embedding dimension.
        texts : list of str
            A list of texts corresponding to each embedding.

        Returns
        -------
        LSHash
            The LSHash object containing the hash tables.
        """
        embeddings = np.array(embeddings)
        if self.input_dim is None:
            self.input_dim = embeddings.shape[1]
        self.lsh = LSHash(hash_size=self.hash_size, input_dim=self.input_dim)

        for emb, txt in zip(embeddings, texts):
            self.lsh.index(emb, extra_data=txt)

        self.texts = texts
        return self.lsh

    def query(self, query_embedding, num_results=5):
        """
        Query the LSH index for similar items to a given query embedding.

        Parameters
        ----------
        query_embedding : np.ndarray
            A 1D array of shape (D,) representing the query embedding.
        num_results : int, optional
            How many nearest neighbors to return, by default 5.

        Returns
        -------
        list of dict
            A list of dictionaries containing 'text', 'distance', and 'similarity' for each retrieved item.
        """
        if self.lsh is None:
            raise ValueError("LSH index has not been built. Please call build_index() first.")

        results = self.lsh.query(query_embedding, num_results=num_results)
        results_info = []
        for result in results:
            neighbor_data = result[0]
            distance = result[1]
            neighbor_text = neighbor_data[1]
            similarity = 1.0 / (1.0 + distance)

            results_info.append({
                'text': neighbor_text,
                'distance': distance,
                'similarity': similarity
            })

        return results_info




Recommends the top 3 drugs for held-out query 50 ("Treatment of Depression / Anxiety disorder / Panic disorder"; actual drug: *Essita 10 Tablet*).

- **ClinicalBERT embeddings** (first cell, output from an earlier session): returns *Antidep 25mg*, *Nildep 25mg* and *Depsol 25mg Tablet* with similarity ≈ 0.41. This is the example shown in the presentation.
- **PubMedBERT embeddings** (second cell, timed): returns the same injection (*Uprise-D3 6L*) three times with similarity ≈ 0.003. Building the index and querying took **4.00 s**.

The mean-pooled PubMedBERT vectors have much larger norms than the [CLS] vectors, so the raw distances and the derived similarity scores can't be compared directly between the two models.

In [ ]:
# Clinical Bert Embeddings
index = 50
if __name__ == "__main__":
    lsh_index = LSHIndex(hash_size=8)
    lsh_index.build_index(embeddings, Names)
    results = lsh_index.query(query_embeddings[index], num_results=3)

print(f"Disease : {Queries[index]}")
Actual_Drug = new_df[new_df['Uses'].str.contains(f'{Queries[index]}', case=False, na=False)]['Name'].iloc[0]
print(f"Actual Drug: {Actual_Drug}")
for rec in results:
    print("Medication:", rec['text'],
          "Distance:", rec['distance'],
          "Similarity:", rec['similarity'])



Disease : Treatment of Depression Treatment of Anxiety disorder Treatment of Panic disorder
Actual Drug: Essita 10 Tablet
Medication: Antidep 25mg Tablet Distance: 1.4345777190794766 Similarity: 0.410748850678755
Medication: Nildep 25mg Tablet Distance: 1.4612743657841711 Similarity: 0.4062935907925065
Medication: Depsol 25mg Tablet Distance: 1.4646088577569096 Similarity: 0.40574389597468224


In [13]:
# PubMed Embeddings

start_time = time()
index = 50
if __name__ == "__main__":
    lsh_index = LSHIndex(hash_size=8)
    lsh_index.build_index(embeddings, Names)
    results = lsh_index.query(query_embeddings[index], num_results=3)

print(f"Disease : {Queries[index]}")
Actual_Drug = new_df[new_df['Uses'].str.contains(f'{Queries[index]}', case=False, na=False)]['Name'].iloc[0]
print(f"Actual Drug: {Actual_Drug}")
for rec in results:
    print("Medication:", rec['text'],
          "Distance:", rec['distance'],
          "Similarity:", rec['similarity'])

end_time = time()
print(f"Time taken for execution: {end_time - start_time:.2f} seconds")

Disease : Treatment of Depression Treatment of Anxiety disorder Treatment of Panic disorder
Actual Drug: Essita 10 Tablet
Medication: Uprise-D3 6L Injection 1ml Distance: 356.7732183463764 Similarity: 0.0027950666755381754
Medication: Uprise-D3 6L Injection 1ml Distance: 356.77326448501617 Similarity: 0.0027950663150848177
Medication: Uprise-D3 6L Injection 1ml Distance: 356.7732663959275 Similarity: 0.002795066300156022
Time taken for execution: 4.00 seconds


### SimHash

**SimHash** works on the raw text rather than on embeddings. Each text is reduced to a binary fingerprint, and the similarity of two texts is 1 − (Hamming distance / fingerprint length). `find_similar_items` compares the query against every sentence in `Data` and keeps matches above a threshold.

In [14]:


def find_similar_items(text_data, query_text, n_items=3, threshold=0.8):

    sim_hash = SimHash()
    query_hash = sim_hash.get_hash(query_text)

    results = []
    for item in text_data:
        item_hash = sim_hash.get_hash(item)
        similarity = 1 - hamming(query_hash, item_hash) / sim_hash.len_hash
        if similarity >= threshold:
            results.append((item, similarity))
    results.sort(key=lambda x: x[1], reverse=True)
    return results[:n_items]


Runs SimHash for the same query 50 with a threshold of 0.9. It returns three drugs whose use text matches the query almost word for word (*Citalin 10*, *Oxetin 20mg*, *Hepinz-S*; similarity 0.94–0.95). Because SimHash re-hashes all 33,909 sentences for every query, it took **44.13 s**.

> SimHash matches surface wording, and the query text appears verbatim in the indexed sentences of drugs with the same use. Its high similarity scores therefore measure something different from the embedding-based scores.

In [15]:
start_time = time()
similar_items = find_similar_items(Data, Queries[50], n_items=3, threshold=0.9)
print(f"Items similar to {Queries[50]}: ")
for item, score in similar_items:
    print(f"\tSimilar Items: {item}\n\tSimilarity: {score:.2f}\n")
end_time = time()
print(f"Time taken for execution: {end_time - start_time:.2f} seconds")

Items similar to Treatment of Depression Treatment of Anxiety disorder Treatment of Panic disorder: 
	Similar Items: Drug Name is Citalin 10 Tablet Which is used for Treatment of Depression Treatment of Anxiety disorder Treatment of Panic disorder
	Similarity: 0.95

	Similar Items: Drug Name is Oxetin 20mg Capsule Which is used for Treatment of Depression Treatment of Panic disorder Treatment of Obsessive -compulsive disorder
	Similarity: 0.94

	Similar Items: Drug Name is Hepinz-S Tablet Which is used for Treatment of Depression Treatment of Anxiety disorder Treatment of Panic disorder
	Similarity: 0.94

Time taken for execution: 44.13 seconds


### FAISS

**FAISS `IndexFlatL2`** performs exact (brute-force) Euclidean nearest-neighbour search over all embeddings. It's highly optimised, so it serves as a fast, exact baseline. The top 2 results are retrieved for queries 50 and 51.

- **ClinicalBERT** (first cell): query 50 returns the same top 2 drugs as LSH (*Antidep*, *Nildep*).
- **PubMedBERT** (second cell): query 50 returns *Citalin 10*, whose listed uses match the query exactly (see the SimHash results), and *Ecitelo 10*. Index construction plus search took **0.16 s**.

> The printed headings use `Queries[query_idx]` with `query_idx` = 0 and 1, but the searched queries are 50 and 51. The labels "Treatment of Allergic conditions" and "Treatment of Dry eyes" are therefore the wrong query texts; the results belong to queries 50 and 51.

In [ ]:
## Clinical Bert Embeddings
embedding_dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(embedding_dimension)
faiss_index.add(embeddings)

k = 2
D, I = faiss_index.search(query_embeddings[50:52], k)

for query_idx, (distances, indices) in enumerate(zip(D, I)):
    print(f"Results for Query {Queries[query_idx]}:")
    for rank, (distance, idx) in enumerate(zip(distances, indices)):
        print(f"  Rank {rank + 1}:")
        print(f"    Name: {Names[idx]}")
        print(f"    Distance: {distance}")



Results for Query Treatment of Allergic conditions:
  Rank 1:
    Name: Antidep 25mg Tablet
    Distance: 1.4345777034759521
  Rank 2:
    Name: Nildep 25mg Tablet
    Distance: 1.4612743854522705
Results for Query Treatment of Dry eyes:
  Rank 1:
    Name: Armotraz Tablet
    Distance: 1.3385933637619019
  Rank 2:
    Name: Anabrez Tablet
    Distance: 1.3415741920471191


In [16]:
## PubMed Embeddings
start_time = time()

embedding_dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(embedding_dimension)
faiss_index.add(embeddings)

k = 2
D, I = faiss_index.search(query_embeddings[50:52], k)

for query_idx, (distances, indices) in enumerate(zip(D, I)):
    print(f"Results for Query {Queries[query_idx]}:")
    for rank, (distance, idx) in enumerate(zip(distances, indices)):
        print(f"  Rank {rank + 1}:")
        print(f"    Name: {Names[idx]}")
        print(f"    Distance: {distance}")

end_time = time()
print(f"Time taken for execution: {end_time - start_time:.2f} seconds")

Results for Query Treatment of Allergic conditions:
  Rank 1:
    Name: Citalin 10 Tablet
    Distance: 100.84402465820312
  Rank 2:
    Name: Ecitelo 10 Tablet
    Distance: 104.79631042480469
Results for Query Treatment of Dry eyes:
  Rank 1:
    Name: Ropark 1 Tablet
    Distance: 81.0308609008789
  Rank 2:
    Name: Ropark 2 Tablet
    Distance: 81.34697723388672
Time taken for execution: 0.16 seconds


## Dataset 1 Emb REC

Loads the saved **Dataset 1** embeddings (4,240 searchable rows and 200 symptom queries) and repeats the LSH and FAISS searches. Here the query is a list of three symptoms and the recommendation is a drug class.

In [ ]:
import numpy as np
embeddings = np.load("/content/drive/MyDrive/BDA_Project/Dataset1/Embeddings/embeddings.npy")
query_embeddings = np.load("/content/drive/MyDrive/BDA_Project/Dataset1/Embeddings/query_embeddings.npy")

print(f"Loaded embeddings shape: {embeddings.shape}")
print(f"Loaded query embeddings shape: {query_embeddings.shape}")

Loaded embeddings shape: (4240, 768)
Loaded query embeddings shape: (200, 768)


### LSH

LSH top 3 for symptom query 50 ("vomiting, sunken_eyes, dehydration"): *Antibiotics* three times, with similarity ≈ 0.29. Many rows in Dataset 1 share the same disease and medication, so the top results are often duplicates.

In [ ]:
# Example usage:
index = 50
if __name__ == "__main__":
    lsh_index = LSHIndex(hash_size=8)
    lsh_index.build_index(embeddings, Names)
    results = lsh_index.query(query_embeddings[index], num_results=3)

    # Print the results
    # Print your heading first:
print(f"Disease : {Queries[index]}")
for rec in results:
    print("Medication:", rec['text'],
          "Distance:", rec['distance'],
          "Similarity:", rec['similarity'])



Disease :  vomiting,  sunken_eyes,  dehydration
Medication: Antibiotics Distance: 2.42644369254496 Similarity: 0.29184778438815057
Medication: Antibiotics Distance: 2.4264488752965523 Similarity: 0.29184734294728154
Medication: Antibiotics Distance: 2.426453218111239 Similarity: 0.2918469730490671


### FAISS

FAISS top 2 for symptom queries 40 and 41.

> As in Dataset 2, the printed query labels come from `Queries[0]` and `Queries[1]`, not from the searched queries 40 and 41.

In [ ]:

embedding_dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(embedding_dimension)

faiss_index.add(embeddings)
k = 2
D, I = faiss_index.search(query_embeddings[40:42], k)
for query_idx, (distances, indices) in enumerate(zip(D, I)):
    print(f"Results for Query {Queries[query_idx]}:")
    for rank, (distance, idx) in enumerate(zip(distances, indices)):
        print(f"  Rank {rank + 1}:")
        print(f"    Data Index: {idx}")
        print(f"    Name: {Names[idx]}")
        print(f"    Distance: {distance}")


Results for Query  high_fever,  blister,  red_sore_around_nose:
  Rank 1:
    Data Index: 491
    Name: NSAIDs
    Distance: 1.6293201446533203
  Rank 2:
    Data Index: 3670
    Name: NSAIDs
    Distance: 1.6293201446533203
Results for Query  continuous_sneezing,  chills,  fatigue:
  Rank 1:
    Data Index: 759
    Name: Laxatives
    Distance: 1.7516059875488281
  Rank 2:
    Data Index: 964
    Name: Laxatives
    Distance: 1.7516059875488281


## Summary

Results recorded in this notebook for Dataset 2 (33,909 drugs, single example queries, Google Colab):

| Technique | Input | Time (index + query) | Example result for "Depression / Anxiety / Panic disorder" |
|---|---|---|---|
| LSH (hash size 8) | PubMedBERT embeddings | 4.00 s | Uprise-D3 6L Injection (×3), similarity ≈ 0.003 |
| LSH (hash size 8) | ClinicalBERT embeddings | not timed | Antidep, Nildep, Depsol 25mg Tablet, similarity ≈ 0.41 |
| SimHash (threshold 0.9) | Raw text | 44.13 s | Citalin 10, Oxetin 20mg, Hepinz-S (drugs with the same listed uses), similarity 0.94–0.95 |
| FAISS `IndexFlatL2` | PubMedBERT embeddings | 0.16 s | Citalin 10, Ecitelo 10 |

- **Speed:** FAISS exact search was fastest, LSH was in between, and SimHash was by far the slowest because it fingerprints the whole corpus for each query.
- **Returned drugs:** with ClinicalBERT embeddings, LSH and FAISS returned the same top drugs for the example query. With PubMedBERT embeddings, FAISS returned a drug whose listed uses match the query exactly, while LSH returned the same injection three times.
- **Limitations:** 200 queries were held out, but quality was only inspected on individual examples. No accuracy or hit-rate metric is computed over the full query set. The "similarity" values of the different techniques use different scales and aren't directly comparable.
- **Future work** (from the presentation): larger datasets, better handling of ambiguous disease descriptions, user feedback, and better query generation.